# ⚖️ LLMs vs Chat Models in LangChain

## Learning Objectives
In this notebook, you will learn:
1. **Two model interfaces** - text-completion LLMs (`BaseLLM`) and chat models (`BaseChatModel`)
2. **A completion LLM** - `OpenAI` takes a string and returns a string
3. **A chat model** - `ChatOpenAI` takes a string or messages and returns an `AIMessage`
4. **How LangChain converts inputs** - a string becomes a `HumanMessage`; a message list becomes one flat string
5. **The same split across providers** - Hugging Face and Gemini classes fall on the same two sides

## Where this fits
`01_LangChain_Inputs_and_Outputs.ipynb` passed both strings and message lists to a chat model. This
notebook shows the two interfaces behind that and what each one does with its input.
`03_Prompt_Templates.ipynb` then builds those inputs from templates.

## Prerequisites
- `01_LangChain_Inputs_and_Outputs.ipynb` (reading)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- `GOOGLE_API_KEY` is optional: without it the Gemini call in Part 5 is skipped

## 🔑 Key Differences

| Feature | LLMs | Chat Models |
|---------|------|-------------|
| **Input** | String | List of Messages (a string is wrapped in one `HumanMessage`) |
| **Output** | String | AIMessage |
| **Examples** | `gpt-3.5-turbo-instruct` | `gpt-4o-mini`, `gemini-2.5-flash` |
| **Use Case** | Simple completions | Conversations, complex tasks |

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and GOOGLE_API_KEY (optional) from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the chat model compared with a completion LLM below
# ============================================================================
from aied_helpers import get_openai_llm

# Named chat_model (not llm) because this notebook contrasts the two interfaces
chat_model = get_openai_llm(model="gpt-4o-mini")

print(f"✅ Chat model initialized: {chat_model.model_name}")

---
## 📖 Part 1: Two model interfaces

In LangChain, the **language model** is the central component. Every model class implements one of
two interfaces:

| Type | Base class | Input | Output |
|------|------------|-------|--------|
| **LLMs** | `BaseLLM` | Text string | Text string |
| **Chat Models** | `BaseChatModel` | List of messages | AIMessage |

> **Note:** In practice, Chat Models are preferred for most applications because they support system prompts, conversation history, and structured outputs.

The cell below checks which side each OpenAI class is on, without calling any API.

In [ ]:
# ============================================================================
# WHICH INTERFACE? check the base class (no model call)
# ============================================================================
from langchain_core.language_models import BaseChatModel, BaseLLM
from langchain_openai import ChatOpenAI, OpenAI


def interface(cls) -> str:
    if issubclass(cls, BaseChatModel):
        return "chat model (messages in, AIMessage out)"
    if issubclass(cls, BaseLLM):
        return "LLM (string in, string out)"
    return "neither"


for cls in (OpenAI, ChatOpenAI):
    print(f"📋 {cls.__name__:<11} -> {interface(cls)}")

---
## 🤖 Part 2: A completion LLM: `OpenAI`

The `OpenAI` class calls OpenAI's **text completion** endpoint (instruct models such as
`gpt-3.5-turbo-instruct`). These models:
- Take a string prompt
- Return a string response
- Have no notion of roles, so they cannot keep conversation context naturally

> **Note:** This is the older API. Most new applications should use a chat model instead.

The provider class is used directly here on purpose: the completion interface is the lesson, and the
course's LLM factory only builds chat models.

In [ ]:
# ============================================================================
# OPENAI LLM (Text Completion Model)
# ============================================================================
# - model: a completion ("instruct") model; chat-only models are rejected here
# - temperature: 0 = deterministic, 1 = more creative
# ============================================================================
completion_llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0)

prompt = "Explain what is Generative AI in 3 bullet points"
response = completion_llm.invoke(prompt)

print("📝 Prompt:", prompt)
print("\n🤖 Response:")
print(response)
print(f"\n📊 Response type: {type(response).__name__}")  # str, not AIMessage

---
## 💬 Part 3: A chat model: `ChatOpenAI`

`ChatOpenAI` calls the **chat completions** endpoint (`gpt-4o-mini` here). Chat models:
- Support system prompts to set behavior
- Can maintain conversation history
- Return structured `AIMessage` objects with metadata

In [ ]:
# ============================================================================
# OPENAI CHAT MODEL (Recommended)
# ============================================================================
# Same prompt string as Part 2 - the chat model wraps it in a HumanMessage
# ============================================================================
response = chat_model.invoke(prompt)

print("📝 Prompt:", prompt)
print("\n🤖 Response:")
print(response.content)
print(f"\n📊 Response type: {type(response).__name__}")
print(f"📊 Token usage: {response.usage_metadata}")

---
## 🔄 Part 4: How LangChain converts inputs

Both interfaces accept both input shapes, because LangChain converts the input to a `PromptValue`
first:

- A **string** sent to a chat model becomes a `StringPromptValue`, whose `.to_messages()` is one `HumanMessage`.
- A **message list** sent to a completion LLM becomes a `ChatPromptValue`, whose `.to_string()` flattens
  the roles into text prefixes.

The first cell shows both conversions without calling a model; the second sends the same message
list to both models.

In [ ]:
# ============================================================================
# INPUT CONVERSION: what each model actually receives (no model call)
# ============================================================================
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompt_values import ChatPromptValue, StringPromptValue

# String -> chat model: one HumanMessage
print("📋 String as messages (what a chat model receives):")
print(StringPromptValue(text=prompt).to_messages())

# Messages -> completion LLM: one flat string with role prefixes
messages = [
    SystemMessage(content="Answer in exactly one sentence."),
    HumanMessage(content="What is Generative AI?"),
]
print("\n📋 Messages as a string (what a completion LLM receives):")
print(ChatPromptValue(messages=messages).to_string())

In [ ]:
# ============================================================================
# SAME MESSAGES, BOTH MODELS
# ============================================================================
completion_reply = completion_llm.invoke(messages)
chat_reply = chat_model.invoke(messages)

print(f"🤖 Completion LLM ({type(completion_reply).__name__}):")
print(completion_reply.strip())
print(f"\n🤖 Chat model ({type(chat_reply).__name__}):")
print(chat_reply.content)

The completion model received `System: ...` and `Human: ...` as ordinary text and simply
continues it, so the system instruction is a hint rather than a role it was trained to follow. The
chat model receives real roles. That difference is why chat models are the default choice.

---
## 🌐 Part 5: The same split across providers

LangChain provides a **unified API** for many providers, and every provider class sits on one of the
two sides:

| Provider | Chat Model Class | LLM Class |
|----------|------------------|-----------|
| OpenAI | `ChatOpenAI` | `OpenAI` |
| Google | `ChatGoogleGenerativeAI` | - |
| Hugging Face | `ChatHuggingFace` (wraps an LLM class) | `HuggingFaceEndpoint`, `HuggingFacePipeline` |
| Anthropic | `ChatAnthropic` | - |

Hugging Face is the clearest case: `HuggingFaceEndpoint` (hosted) and `HuggingFacePipeline` (local) are
completion-style LLMs, and `ChatHuggingFace` wraps either one to give it the chat interface. Live
Hugging Face calls are in `modules/01_getting_started/notebooks/07_Open_Source_LLMs_with_LangChain.ipynb`.

In [ ]:
# ============================================================================
# PROVIDER CLASSES: which interface each one implements (no model call)
# ============================================================================
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline

for cls in (OpenAI, HuggingFaceEndpoint, HuggingFacePipeline,
            ChatOpenAI, ChatHuggingFace, ChatGoogleGenerativeAI):
    print(f"📋 {cls.__name__:<22} -> {interface(cls)}")

In [ ]:
# ============================================================================
# OPTIONAL: GOOGLE GEMINI - another chat model, same AIMessage out
# ============================================================================
# Runs only when GOOGLE_API_KEY is set in .env
# Current models: https://ai.google.dev/gemini-api/docs/models
# ============================================================================
import os

if os.getenv("GOOGLE_API_KEY"):
    gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    response = gemini.invoke(prompt)
    print(f"📊 Response type: {type(response).__name__}")
    print("🤖 Gemini Response:")
    print(response.content)
else:
    print("⏭️ GOOGLE_API_KEY not set - Gemini call skipped")

---
## 📝 Summary

In this notebook, we learned:

### 1. Two model interfaces
- Every LangChain model is either a `BaseLLM` (string in, string out) or a `BaseChatModel` (messages in, `AIMessage` out)

### 2. A completion LLM
- `OpenAI` calls the text-completion endpoint with instruct models and returns a plain `str`

### 3. A chat model
- `ChatOpenAI` returns an `AIMessage` with `.content` and token usage - the recommended interface

### 4. How LangChain converts inputs
- A string sent to a chat model becomes one `HumanMessage`
- A message list sent to a completion LLM is flattened into `System: ...` / `Human: ...` text, so roles are lost

### 5. The same split across providers
- `ChatGoogleGenerativeAI` and `ChatHuggingFace` are chat models; `HuggingFaceEndpoint` and `HuggingFacePipeline` are LLMs
- Swapping one chat model for another keeps the same call shape and the same `AIMessage` output

### Next Steps
- **`03_Prompt_Templates.ipynb`**: build these strings and message lists from reusable templates